In [0]:
import requests
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, IntegerType, ArrayType, MapType

url = "https://dummyjson.com/products?limit=100"
response = requests.get(url).json()

data = response['products']

# Define explicit schema to handle mixed types
schema = StructType([
    StructField("id", IntegerType(), True),
    StructField("title", StringType(), True),
    StructField("description", StringType(), True),
    StructField("category", StringType(), True),
    StructField("price", DoubleType(), True),
    StructField("discountPercentage", DoubleType(), True),
    StructField("rating", DoubleType(), True),
    StructField("stock", IntegerType(), True),
    StructField("tags", ArrayType(StringType()), True),
    StructField("brand", StringType(), True),
    StructField("sku", StringType(), True),
    StructField("weight", IntegerType(), True),
    StructField("dimensions", MapType(StringType(), DoubleType()), True),
    StructField("warrantyInformation", StringType(), True),
    StructField("shippingInformation", StringType(), True),
    StructField("availabilityStatus", StringType(), True),
    StructField("reviews", ArrayType(MapType(StringType(), StringType())), True),
    StructField("returnPolicy", StringType(), True),
    StructField("minimumOrderQuantity", IntegerType(), True),
    StructField("meta", MapType(StringType(), StringType()), True),
    StructField("images", ArrayType(StringType()), True),
    StructField("thumbnail", StringType(), True)
])

df = spark.createDataFrame(data, schema=schema)
# display(df)

In [0]:
# Save to Unity Catalog table
# df.write.mode("overwrite").saveAsTable("bronze_products")

In [0]:
%sql
-- Select * from bronze_products

In [0]:
# df.write.mode("overwrite").format("parquet") \
# .save("/FileStore/retail_inventory/bronze_products")

In [0]:
df_clean = df.dropDuplicates().fillna({
    "stock": 0,
    "price": 0
})
# # display(df_clean)
# # df.write.mode("overwrite").format("parquet").save("/Workspace/Users/aditya.rana.datascience@gmail.com/Retal_Inventory_managment/bronze/bronze_products")

In [0]:
low_stock_df = df.filter("stock < 20") 
overstock_df = df.filter("stock > 100")

In [0]:
low_stock_df.write.mode("overwrite").saveAsTable("low_stock_alerts")
overstock_df.write.mode("overwrite").saveAsTable("overstock_alerts")

In [0]:
df.write.mode("overwrite").saveAsTable("bronze_products")
# df.write.format("parquet").save("/Workspace/Users/aditya.rana.datascience@gmail.com/Retal_Inventory_managment/bronze/bronze_products")

In [0]:
from pyspark.sql.functions import current_timestamp

df = df.withColumn("ingestion_time", current_timestamp())